# 加载 File Directory

目录加载器本身通常不解析内容，它负责遍历文件、筛选扩展名，并把每个文件分派给对应解析函数。下面用 `pathlib.rglob()` 实现一个透明的目录加载流程。

In [ ]:
from collections.abc import Iterator
from pathlib import Path
from bs4 import BeautifulSoup
from langchain_core.documents import Document

ASSET_DIR = Path.cwd() / "系统学习" / "6-LangChain中的RAG" / "asset" / "load"

def read_text_with_fallback(path: Path) -> tuple[str, str]:
    for encoding in ("utf-8", "gbk"):
        try:
            return path.read_text(encoding=encoding), encoding
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError("unknown", b"", 0, 1, f"无法解码 {path}")

def load_simple_file(path: Path) -> Document:
    suffix = path.suffix.lower()

    if suffix in {".txt", ".md"}:
        text, encoding = read_text_with_fallback(path)
    elif suffix == ".html":
        html = path.read_text(encoding="utf-8")
        text = BeautifulSoup(html, "html.parser").get_text("\n", strip=True)
        encoding = "utf-8"
    else:
        raise ValueError(f"不支持的文件类型：{suffix}")

    return Document(
        page_content=text,
        metadata={
            "source": str(path),
            "format": suffix.lstrip("."),
            "encoding": encoding,
        },
    )

def load_directory(directory: Path) -> Iterator[Document]:
    supported = {".txt", ".md", ".html"}
    for path in sorted(directory.rglob("*")):
        if path.is_file() and path.suffix.lower() in supported:
            yield load_simple_file(path)


In [ ]:
documents = list(load_directory(ASSET_DIR))
print(f"共加载 {len(documents)} 个文档")
for document in documents:
    print(document.metadata, len(document.page_content))


真实项目通常还会配置忽略规则、最大文件大小、符号链接策略、错误处理和并发读取。PDF、DOCX 等格式可以继续加入分派表，而不是把所有格式塞进一个巨大的判断函数。